# Annotation
This notebook is used to annotate LLM's reasoning traces with our taxonomy and validate the performance of the LLM-annotator.

Implements the annotation pipeline of Section 3.3 / Appendix A.5 (chunking + grounded-example extraction, Algorithm 1) using the ten-strategy taxonomy (Table 1). Validated against manual annotations in Table 14 (precision 0.97 / recall 0.95 macro-averaged, cited in Section 4.2).

In [ ]:
import os
import json
import glob
import re
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm import tqdm

from openai import OpenAI

import pandas as pd
from pathlib import Path

from src.datasets import get_or_create_dataset
from src.model_configurations import gpt_4_1, gpt_4_1_mini_det_config, gpt_5_mini_config
from src.prompt_util import prompt_openai

load_dotenv()

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)
print(f"We have {len(sciq_dataset)} SciQ questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset,
    "sciq_data": sciq_dataset,
}

equality_model_config = gpt_4_1_mini_det_config
expert_equality_model_config = gpt_5_mini_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))

In [ ]:
EEDI_TAXONOMY = """
<INTER>
Definition: Reasoning about the task instructions or requirements — what the question asks for and what counts as valid answers.  
Rules:
- Only mark when the expert revisits the task description and subsequently tries to gain clarity about the task itself. 
- Do NOT mark execution steps, calls to produce output, or listing candidates (e.g., "I'll produce:", "Let's do:", "Distractor1: 0.4<INST>"). 
Examples:
- "We are given the question: ..."
- "However<RECON>, the task is to generate three incorrect distractors, not the correct answer<INTER>"

<LINK>
Definition: Naming a conceptual axis or category from which candidate distractors are drawn.
Rules:
- Mark only the axis-preamble in an "axis: item1, item2, ..." enumeration. Each named item still gets <INST>.
- Do NOT mark output-format/rendering meta (untagged): "options should be in decimal form", "options formatted similarly to the correct answer", "I should output exactly:", "use the template".
- Do NOT mark error-category headers like "Common mistakes:" or "Common errors students make:" — these introduce ERR_DESC content.
- Rare in math traces; most distractors come from error simulation (ERR_DESC/ERR_SIM), not conceptual-category enumeration.

<CORR>
Definition: Correct computation or reasoning toward the correct solution for the question.
Rules:
- Mark whenever correct reasoning or the correct answer is referenced.
- If correct reasoning and errors are discussed together, mark both.
Examples:
- "2 ÷ 1/5 = 10<CORR>"
- "Multiplying both sides by 4 gives 20 = k<CORR>, but a student might only multiply the numerator<ERR_DESC>"

<ERR_DESC>
Definition: High-level verbal description of a common mistake or misconception.
Rules:
- Mark every description of an error.
Examples:
- "A common mistake is forgetting to flip the fraction<ERR_DESC>"
- "46 <INST> (forgetting to add 2)<ERR_DESC>"
- "(x,y)=(-2,15)<INST> [from sign error<ERR_DESC>]"
- "Mis-handling the negative<ERR_DESC>: -10 + 8 <ERR_SIM> = 2<INST>"

<ERR_SIM>
Definition: Explicitly simulating incorrect reasoning.
Rules:
- Mark when the expert simulates an incorrect calculation.
- Single incorrect equations can be marked if they represent erroneous reasoning.
- Mark the final incorrect outcome with <INST>.
- ERR_DESC = a high level error description; ERR_SIM = a specific execution of an error
Examples:
- "5 - 2 = 3, then add 1 = <ERR_SIM> 4<INST>"
- "9 + 3 = 12, write down 2, forget to carry the 1… final result <ERR_SIM> 82<INST>"
- "Convert the fraction incorrectly <ERR_DESC>: compute: 1 2/3 <ERR_SIM><INST>"

<INST>
Definition: Any incorrect outcome (number, symbol, expression).
Rules:
- Mark every candidate, even if later rejected.
- Mark candidate values even when they appear inside task interpretation or reconsideration spans, as long as they name concrete answer options
- Each value in an enumeration of candidates is marked separately; enumeration markers like 1., 2., 3. are NOT tagged.
Examples:
- "0.4<INST>, 0.1<INST>, 2.5<INST>"
- "Possible answers could be Alice <INST>, Bob <INST>, etc"
- "980<INST> might work"
- "(x,y)=(-2,15)<INST> [from sign error<ERR_DESC>]"

<PLAUS>
Definition: Judgment of how likely a student would choose an error or candidate.
Rules:
- Mark plausibility comparisons or checks for incorrectness.
- If also about final set, mark both PLAUS and CURATE.
Examples:
- "0.4<INST> is more plausible than 0.1<INST><PLAUS>"
- "The student forgets to add?<ERR_DESC> Plausibly<PLAUS>"
- "0.4<INST> is not a good distractor<PLAUS>"
- "But is a student going to make that mistake?<PLAUS>"

<DISCR>
Definition: Verifying a candidate is incorrect / distinct from the correct answer (or rejecting one that's too close).
Rules:
- Mark the verification act, not the candidate itself.
- Triggers: distinctness checks ("all different", "X is correct, so we need incorrect"), per-candidate "but it's not the answer" tail-clauses (mid-sentence split), "too close to correct" rejections.
- Distinct from PLAUS (would a student pick it?) and CURATE (final-set quality).
- Do NOT fire on general "doesn't make sense" / "not sure" dismissal or on shape-based rejection ("not an integer"). DISCR asserts incorrectness or insufficient distinctness from the correct answer.
Examples:
- "0.256 is correct, so we need incorrect ones.<DISCR>"
- "8/3 ≈ 2.66 is too close to 2.828 and 2.<DISCR>"

<CURATE>
Definition: Evaluation or selection of the final set of distractors (coverage, diversity, redundancy).
Rules:
- Only mark when reasoning explicitly concerns the final set.
- Otherwise, mark PLAUS.
Examples:
- "Keep 0.4<INST> and 2.5<INST>, drop 0.1<INST> to cover error types<CURATE>"
- "0.4<INST> seems plausible<PLAUS>, keep that<CURATE>"

<RECON>
Definition: Reconsideration of a previous interpretation, candidate, plausibility judgment, or curation decision.
Rules:
- Place <RECON> immediately after the cue word indicating reconsideration.
- Marks the act of reconsidering, not the outcome.
- Common cues: "actually", "alternatively", "instead", "however", "but wait", "on second thought", "reconsider"
Examples:
- "Actually<RECON>, ..."
- "Alternatively<RECON>, 980<INST> could work<PLAUS>"
- "On second thought<RECON>, that distractor is not likely<PLAUS>"
"""


SCIENCE_TAXONOMY = """
# Output format (applies to every emitted tag)
- Each label is its own angle-bracket tag: <PLAUS><DISCR>. NEVER combined: <PLAUS+DISCR>, <PLAUS_DISCR>, <PLAUS/DISCR>. The `+` and `-` prefixes inside angle brackets are reserved for manual user corrections (false-negative / false-positive) and must NOT appear in annotator output.

# Cross-cutting rules

CC1. **Tag every occurrence.** Re-tag <INST>, <ERR_DESC>, <PLAUS>, <DISCR>, <CORR>, <RECON> on every mention — including repeated mentions of the same candidate, misconception, or pivot inside a deliberation loop. Co-occurrence patterns are used in downstream analysis; first-mention-only tagging is wrong. Tag density is not a concern.

CC2. **Stack only when each tag definitely applies.** Stack multiple tags on the same span only when *each* tag genuinely applies. Do NOT default-stack <PLAUS><DISCR> on every candidate justification — co-tag both only when the same span explicitly does both jobs (attractiveness AND incorrectness, e.g., "sounds like Y<PLAUS> but is actually Z<DISCR>").

CC3. **Output-formatting talk is untagged.** Any sentence whose primary content is *how the final distractor list should appear, be phrased, or be laid out* is left untagged, regardless of trace position. Triggers:
- References to template/format: "the template", "apply the template", "for the template", "let's see the template", "to adhere to the template", "looking back at the template", "the requested/required/specified format", "in the format", "format the output", "output it", "I'll output…", "format it exactly like that", "let me write them in the format", "I need to fill in the distractors", "fill in the distractors".
- Layout / phrasing instructions: "list them out", "on its own line", "as a single word", "as complete phrases", "should be names", "the answers might be short phrases", "the answers should be a category / single words / short phrases", "in multiple-choice questions/exams the options are…", "I'll provide the answers themselves, not explanations", "I have to take the question as given".
- Selection-strategy meta about output choices ("I'll go with three distinct ones", "let me phrase them as X") only when the sentence is *about phrasing/format*, not selection — pure selection is <CURATE>.
- Section headers: **Format:**, **Output Format:**, **Formatting:**, **Drafting the Output:**, **Final Polish:**, **Format Output:**.
- Bare template skeleton: "Distractor1: ... Distractor3:", "[Template]".

When in doubt, ask: *"Does this sentence describe the final-list rendering — template, layout, or what form the answers take?"* If yes → untagged. Real examples that must be untagged (NOT INTER, NOT LINK, NOT CURATE):
- "Apply the requested template."
- "I'll write each distractor on a separate line with the prefix."
- "The answers should be single words or short phrases."
- "In the template, it's just the distractor answers, so I'll keep them as organ names."
- "I need to fill in the distractors."
- "Let's see the template:"
- "However, to adhere to the template, I need to output Distractor1, Distractor2, Distractor3."
- "Looking back at the user's template…"
- "I have to take the question as given."
- "I need to phrase these as answers to 'X'."
- "In multiple-choice exams the options are usually short phrases."

CC4. **Rhetorical context disambiguates.** The same factual statement gets different tags by *function*, not surface form. Example: "Deposition is gas to solid." → <CORR> when committing to / grounding the correct answer; → <LINK> when invoked to define the distractor design space ("Since deposition is gas to solid, liquid would be the wrong choice.").

CC5. **Place tags at clause boundaries, not at sentence ends.** When a sentence contains multiple functional spans, insert each tag at the END of its sub-clause. Do NOT collapse multiple tags onto the trailing period.
- `X<INST> is tempting because Y<PLAUS>, but Z<DISCR>.` — PLAUS at end of attractiveness clause, DISCR at end of incorrectness clause. NOT `X<INST> is tempting because Y, but Z.<PLAUS><DISCR>`.
- `X<INST> – because students might confuse it with Y<ERR_DESC>, but it's actually different.<DISCR>` — ERR_DESC at end of misconception clause, DISCR at end of contrast clause.
- `Distractors should be related to X<LINK> but misclassify Y.<INTER>` — LINK on design-space clause, INTER on task-interpretation clause.
- Real example (STI distractor): `"Viral STIs"<INST> – a common mistake because many STIs are viral<PLAUS>, but these are not.<DISCR>`
- Real example (misconception): `"Incurable STIs"<INST> – because people might think STIs are incurable<ERR_DESC>, but these are generally curable.<PLAUS>`
- **Dash-introduced distractor template:** `Distractor{N}: X<INST> – {attractiveness clause}<PLAUS>, but {incorrectness clause}.<DISCR>` — PLAUS goes at the END of the attractiveness clause (right before the `but`/comma), DISCR goes at the END of the incorrectness clause. Three real examples (greenhouse-gas distractors):
  - `Distractor1: Oxygen (O2)<INST> – common in the atmosphere<PLAUS> but not a greenhouse gas.<DISCR>`
  - `Distractor2: Nitrogen (N2)<INST> – the most abundant gas in the atmosphere<PLAUS>, but it doesn't trap heat.<DISCR>`
  - `Distractor3: Carbon monoxide (CO)<INST> – often confused with CO2<PLAUS>, but it's not a significant greenhouse gas.<DISCR>`
- **Bullet-introduced distractor template:** `**X**<INST>: {attractiveness clause}<PLAUS>, but {incorrectness clause}.<DISCR>` — same split. Example: `**Boron**<INST>: Transition metals can form borides, which are also hard materials<PLAUS>, but they are not carbides.<DISCR>`

Co-stack <PLAUS><DISCR> on the same span only when one span explicitly does both jobs in lockstep ("sounds like Y but is actually Z"). If you can name distinct sub-clauses, split.

CC6. **Pivot cues at sentence start ALWAYS trigger <RECON> immediately after the cue — no exceptions.** Before tagging anything else in a sentence, check whether the sentence starts with a pivot cue. If yes, place `<RECON>` immediately after the cue word/phrase and its trailing punctuation, THEN continue tagging the rest of the sentence on its own per its category's rules. The rest of the sentence's content (LINK / INST / PLAUS / DISCR / CURATE) does NOT replace RECON — both fire.

Trigger words/phrases (sentence-start, ALWAYS RECON):
- `Instead,` / `Instead of …,`
- `Alternatively,`
- `Another idea:` / `Another option:` / `Another thought:` / `Another approach:`
- `Wait,` / `But wait,`
- `Actually,` / `Actually:`
- `Hmm,`
- `On second thought,`
- `Let me reconsider` / `Let me rethink`
- `*Self-Correction:*` / `Self-Correction:` (italic or plain)
- `On reflection,` / `Rethinking,`

Real examples that v10 missed and v11 must catch:
- `Alternatively<RECON>, I could use silicon<INST>, as silicon carbide is well-known<PLAUS>, but it's not an interstitial carbide; it's a covalent carbide.<DISCR>`
- `Alternatively<RECON>, "Energy"<INST> is a good distractor because atoms do gain/lose energy<PLAUS>, but it doesn't result in a charged ion.<DISCR>`
- `Alternatively<RECON>, in museums, biological specimens are often cataloged using systems like the Linnaean taxonomy<CORR>.`

If you find yourself emitting tags for a sentence that begins with one of the pivot cues above and you have NOT placed `<RECON>` on the cue, that is a bug — go back and add it.

# Tags

<INTER>
Definition: The expert engages with what the task or question is asking — interpreting the source question, restating the goal, breaking down key phrases, or articulating what kind of answer is expected.
Rules:
- Mark every span where the expert engages with what the task/question is, including verbatim restatements and paraphrases.
- Switch to <CORR> as soon as the span moves from understanding the task to solving it.
- NOT scaffolding: section/step headers ("Analyze the Question:", "Brainstorm Distractors:", "Final Selection:", "Identify the Correct Answer:") are organizational labels, not task understanding.
- NOT output formatting (see CC3) — leave that untagged. Sentences like "Looking back at the template", "I have to take the question as given", "I need to phrase these as answers to X", "I have to take the question as given" are template/format meta, NOT INTER.
- End-of-trace selection statements in any phrasing ("let me pick three that fit", "I'll go with…", "these work") are <CURATE>, not INTER.
Examples:
- "The question is about photosynthesis and asks for the primary product.<INTER>"
- "I need to generate three incorrect distractor answers for a multiple-choice exam.<INTER>"

<CORR>
Definition: States, deduces, or grounds the correct answer to the MCQ, including factual recall used to identify it.
Rules:
- Mark spans that answer "What or why is the correct answer?".
- Hedging meta toward committing to the answer ("the question might be referring to a specific one", "to be safe I'll consider…", "since the question doesn't specify, I'll keep it general", "in exams they might expect…") is part of CORR-grounding work — tag <CORR>, not untagged.
- Per CC4, the same factual statement is <CORR> when grounding the answer but <LINK> when invoked to define the design space.
- **Inline-term CORR:** when the correct answer is named explicitly inside prose (often quoted), place <CORR> on the **specific term span**, not just on the surrounding sentence. Examples:
  - `Since the correct answer is "bacterial,"<CORR> I can create distractors based on other types:<LINK>` — CORR on the quoted term, LINK on the design-space clause that follows (per CC5).
  - `The thighbone is the femur<CORR>.` — CORR on the term itself.
  - In an inline candidate list, the correct-answer term gets both <INST> and <CORR>: `Common types of energy include kinetic<INST><CORR>, potential<INST>, thermal<INST>, …`
Examples:
- "The correct answer is mitochondria because they produce ATP via oxidative phosphorylation.<CORR>"
- "Glucose is the primary product of photosynthesis.<CORR>"

<LINK>
Definition: Establishes the conceptual / categorical relationship that distractors should share with the correct answer — the design space, with no specific candidate named.
Rules:
- Mark spans that answer "What kind of things should distractors generally be?".
- Specificity test: if you can replace the noun with "such things" and the sentence still reads as a category-level criterion, it is LINK. Once a specific candidate appears, switch to <PLAUS> / <DISCR> / <ERR_DESC>.
- Candidate enumeration: when a bullet/list enumerates concrete terms, tag each named term as <INST>, not the bullet as <LINK>. See INST → "Candidate enumeration patterns" for the full pattern set. In particular, for an *axis-then-items* bullet (e.g., `- By causative agent: bacterial, viral, parasitic, fungal.`), the <LINK> belongs on the **axis preamble** ("By causative agent") and each item gets <INST> — do NOT trail <LINK> at the end of the bullet.
- LINK is about *what kind of thing* a candidate is, NOT *how the candidate text is rendered* in the final list — phrasing/length constraints are untagged (CC3). Sentences like "I need three distractors", "for the template", "the answers should be a category", "in such questions, the answers are given as single words or short phrases", "I should make them sound formal", "I need to fill in the distractors" are template/format meta, NOT LINK.
- **Brainstorm-introduction phrases are untagged, NOT LINK.** Transitional phrases that *introduce* a candidate-list (the candidates appear in subsequent lines/bullets) are scaffolding — neither LINK nor CURATE. Examples that must be UNTAGGED:
  - "Possible distractors could include:"
  - "Possible incorrect answers could include:"
  - "Possible related terms that students might mix up include:"
  - "Possible animal groups to consider:"
  - "Distractors are wrong options that might seem plausible to someone who doesn't know the correct answer." (generic distractor-defining meta)
  - "But I need to make sure they are incorrect." / "but distractors should be wrong." (process meta)
  - "I should ensure they are distinct and likely to be chosen by students who misunderstand the process." (set-quality meta with no candidate type named)
  - "Let's list them out." (output-format scaffolding, also CC3)
  These phrases describe *that* a brainstorm is happening, not what kind of thing the candidates should be. LINK fires only when the sentence specifies a candidate *type* or *property* (e.g., "Distractors should be other types of bonds<LINK>", "These should be flat bones from the body<LINK>").
Examples:
- "Distractors should be other types of bonds not primarily formed to give atoms a more stable electron arrangement.<LINK>"
- "These should be structures or systems related to movement but not the primary ones.<LINK>"

<INST>
Definition: A specific value, term, or concept named as a concrete answer-shaped option in any phase of the trace — explicit candidacy, brainstormed candidate, or enumerated within taxonomic brainstorming.
Rules:
- Each value in an enumeration is tagged separately. Enumeration markers (1., 2., 3.) are not tagged.
- A candidate that turns out to be the correct answer is still <INST> if it was first considered as a candidate (e.g., "Oxygen<INST> (correct)<CORR>"). Vague pseudo-candidates ("other properties of blood") in a candidate-bullet are also INST when the bullet is functionally a candidate slot.
- **Scope broadened to taxonomic brainstorming:** A term gets <INST> whenever it is named as a concrete answer-shaped option, NOT only when explicitly drafted as "Distractor N: X". Enumerated terms inside a taxonomic-axis bullet during distractor brainstorming (e.g., listing "bacterial, viral, parasitic, fungal" under "By causative agent") all get <INST> — they are candidate classes the writer is considering, even before any is formally drafted. Test: would this term plausibly become a distractor or distractor-class? If yes → <INST>.
- **Candidate enumeration patterns — each named term gets <INST>:**
  - *Inline ideas:* `*Idea N: Term.*<INST>` — Term, not the bullet, gets <INST>.
  - *Bold sub-header bullets:* `- **Term:** explanation…` where the list enumerates candidates being evaluated. Term gets <INST> immediately after the bold-close colon. Example:
    - `- **Arteries:**<INST> Carry oxygenated blood away from the heart …<ERR_DESC>`
    - `- **Veins:**<INST> Carry deoxygenated blood …<ERR_DESC>`
  - *Plain sub-header bullets:* `- Term: explanation…` (no bold) — same rule. Example: `- Amplitude:<INST> distance from rest position to crest/trough …<DISCR>`.
  - *Inline comma list of candidates:* "the three main types are A, B, and C" — every named term gets <INST>, **including the correct-answer term** (which then also carries <CORR>). Example: `The three main types are arteries<INST>, veins<INST>, and capillaries<INST><CORR>.` — `capillaries` is NOT skipped just because it's the right answer.
  - *Axis-then-items bullet (taxonomic brainstorming):* `- {axis-name}: {term1}, {term2}, {term3}.` — the axis-name preamble gets <LINK> and each named term gets <INST>. NO trailing <LINK> at the end of the bullet. Examples:
    - `- By causative agent<LINK>: bacterial<INST>, viral<INST>, parasitic<INST>, fungal<INST>.`
    - `- In the lower limb<LINK>: tibia<INST>, fibula<INST>, patella<INST>.`
    - `- Concepts related to chemistry/matter<LINK>: Compounds<INST>, Mixtures<INST>, Molecules<INST>, Atoms<INST>.`
    - When a bullet is *only* a category description with no concrete terms (e.g., `- By other categories like "systemic" or "localized."`), tag the whole bullet as <LINK> — no INST.
  - Heuristic: if the list *evaluates* the named items as potential answers (DISCR/PLAUS/ERR_DESC follows them), it is candidate-enumeration → tag each term <INST>. If the list is purely background exposition with no evaluation, it is not enumeration.
Examples:
- "Possible distractors: apples<INST> (because they are red<PLAUS>)"
- "Radiation removing things other than electrons<ERR_DESC> (e.g., protons<INST>, neutrons<INST>)."

<PLAUS>
Definition: Establishes, justifies, or assesses why a *specific candidate* would be tempting — similarity, familiarity, lexical overlap, conceptual closeness, commonness.
Rules:
- Requires a specific candidate. General "distractors should be plausible" is <LINK>, not PLAUS.
- A bare statement of similarity / closeness / commonness suffices (no action verb required, unlike ERR_DESC).
- **Common-misconception sentences stack ERR_DESC+PLAUS on the same span.** When a sentence describes a "common misconception / common mix-up / common confusion" for a specific candidate, the misconception's *commonness* itself establishes plausibility — co-tag both `<ERR_DESC>` and `<PLAUS>` at the end of the span (no sub-clause split needed). Examples:
  - `Glucose<INST> – a common misconception that respiration produces glucose, when it actually consumes it.<ERR_DESC><PLAUS>`
  - `Oxygen<INST> – a common mix-up where students think respiration produces oxygen like photosynthesis does.<ERR_DESC><PLAUS>`
- Co-tag PLAUS+DISCR only when both functions are explicit in the same span (CC2). Prefer splitting at clause boundaries (CC5) over stacking.
- **Dismissive scoping is not PLAUS.** When a brainstorming bullet *excludes* a term from the candidate pool because it is out-of-scope or low-relevance ("but not in the leg", "but these are not typically confused with the thighbone", "but they're not really considered as STIs in the same way"), the dismissal is a candidate-pool filter, NOT a per-candidate plausibility assessment. Leave such tail clauses untagged. PLAUS fires only when the writer is evaluating an *already-named distractor candidate* against the question.
Examples:
- "mitosis<INST> (a common confusion<PLAUS>)"
- "This may be too obscure to be convincing.<PLAUS>"

<DISCR>
Definition: Reasons about a *specific candidate's* incorrectness or contrast with the correct answer.
Rules:
- Requires a specific candidate's incorrectness. General design goals ("distractors should be wrong", "I need to make sure these are incorrect") are <LINK>, not DISCR. Whole-set sanity checks ("Do these all answer the prompt? Yes") are <CURATE>, not DISCR.
- DISCR applies to *candidate evaluations* — why a brainstormed term is wrong as an answer to the question. Format/selection-strategy meta ("this won't match the template", "too similar to another distractor", "it's better to avoid", "so it might not match the question format", "these are specific rocks, not types") is untagged (or <CURATE> if it commits to a selection), NOT DISCR.
- **Dismissive scoping is not DISCR.** Tail clauses that *exclude* a term from the candidate pool ("but not in the leg", "but these are not typically confused with X", "but it's not a long bone") are candidate-pool filters, NOT per-candidate incorrectness evaluations. Leave them untagged. DISCR fires only when the writer is evaluating an *already-named distractor candidate* as an answer to the question itself.
- Co-tag PLAUS+DISCR only when both functions are explicit in the same span (CC2). Prefer splitting at clause boundaries (CC5) over stacking.
Examples:
- "Reptiles have scales and dry skin, which is not permeable like amphibians.<DISCR>"
- "It sounds plausible because of x<PLAUS> but is wrong due to y.<DISCR>"

<ERR_DESC>
Definition: A high-level description of a student's cognitive error or misconception.
Rules:
- The span MUST contain a cognitive action verb (confuse, mix up, overgeneralize, forget, misapply, assume, associate, mistake, …). Without one, the span is <PLAUS> or <DISCR>, not ERR_DESC.
- Impersonal phrasing counts when an action verb is present ("this is commonly confused with X").
- Pure factual statements ("X is bacterial", "Y is the opposite of Z") do NOT count.
Examples:
- "Common confusions might involve mixing up radiation types with other physics concepts.<ERR_DESC>"
- "A student might associate mitosis with cell growth and forget it produces two identical daughter cells.<ERR_DESC>"

<ERR_SIM>
Definition: A step-by-step procedural simulation of a misconception, showing how the wrong reasoning unfolds.
Rules:
- ERR_DESC = high-level description of the error; ERR_SIM = a specific procedural execution of it.
Example:
- "A student might think: photosynthesis needs light, mitochondria use light too<ERR_SIM>, so the answer is mitochondria<INST>."

<CURATE>
Definition: Selects, refines, organizes, or finalizes the distractor set — improves overall set quality.
Rules:
- Mark any commitment to the final selection regardless of phrasing — informal commits ("I'll go with…", "Let's go with…", "these work", "let me finalize"), italic-emphasized standalone-line commits like `*Let's go with the most distinct categories.*<CURATE>` or `*Let's go with the simplest, most effective distractors.*<CURATE>`, end-of-trace selection ("let me pick three that fit"), bare list outputs (place <CURATE> on the closing line of the Distractor1/2/3 block), set-level sanity checks and quality assertions including templated "Ensure / make sure / verify that the distractors are X (and not Y)" sentences at the end of a step ("Ensure distractors are scientifically accurate bone names and are not overly obscure.<CURATE>" — NOT INTER; "Make sure these are clear and concise.<CURATE>" — only when it's a *quality* criterion, not a phrasing/format directive which is CC3-untagged), and "Do these all answer the prompt? Yes.<CURATE>".
- Refining / Decision sub-headers ("*Refining the list:*", "*Refining Selection 3:*", "*Decision:*", "*Refined Selection:*", "*Final Decision:*", "let me refine") are CURATE — tag the header itself, even when bare. These are NOT RECON cues; they refine an already-working selection.
- Phrases that *introduce* a brainstorm or enumeration of candidates ("Possible incorrect processes:", "Let me list some greenhouse gases:", "Distractor ideas:") are NOT CURATE — they belong to the generation phase. CURATE applies only when *selecting / refining* among already-generated candidates.
Examples:
- "Let's select the three strongest ones that sound plausible but are incorrect.<CURATE>"
- "Final review: do these clearly contrast with the correct answer? Yes.<CURATE>"
- Bare final-list block: each named term gets `<INST>` (per INST rules); place `<CURATE>` on the closing line of the block. Do NOT leave the terms untagged.
    "Distractor1: Tibia<INST>
    Distractor2: Humerus<INST>
    Distractor3: Fibula<INST><CURATE>"
    "Distractor1: Oxygen<INST>
    Distractor2: Nitrogen<INST>
    Distractor3: Carbon monoxide<INST><CURATE>"

<RECON>
Definition: The expert challenges or reconsiders a previous assumption, interpretation, candidate, plausibility judgment, or curation decision.
Rules:
- Marks the act of reconsidering, not the outcome. New content following <RECON> is tagged on its own per its category's rules.
- Pivot cues: "wait", "but wait", "actually", "alternatively", "instead", "however", "on second thought", "reconsider", "let me reconsider", "another idea", "let's try another angle", "self-correction" / "*Self-Correction:*" / "let me self-correct", "on reflection", "rethinking".
- Placement: <RECON> goes IMMEDIATELY AFTER the cue word at sentence start, never trailing the sentence. Correct: "Instead<RECON>, I'll use rock types from different categories.<LINK>". Wrong (RECON missing): "Instead, I'll use rock types…<LINK>". Wrong (RECON pushed to end): "Upon second thought, … colon.<RECON>".
- Bare-cue placement: For bare cues at sentence start ("Wait,", "Actually,", "Hmm,", "On second thought,", "Another thought,"), place <RECON> immediately after the cue word and its trailing comma — NOT at the end of the sentence. Correct: `Wait,<RECON> the question asks about X.` Wrong: `Wait, the question asks about X.<RECON>`.
- "Instead" at sentence start ALWAYS triggers <RECON>. So does "Instead of …,". Do not skip it just because the rest of the sentence carries other content (LINK / INST / CURATE) — RECON tags the pivot act; the rest of the span is tagged on its own.
- "Alternatively," / "Another idea:" / "Another option:" / "Another thought:" at sentence start ALSO ALWAYS trigger <RECON> immediately after the cue, regardless of what the rest of the sentence carries. Examples (do NOT skip):
  - `Alternatively<RECON>, they might think it's by their empirical formula<ERR_DESC>.`
  - `Another idea:<RECON> Use gases that are greenhouse gases but not the primary one<INST>.`
- "Refining" / "let me refine" is NOT a RECON cue — those are CURATE.
Examples:
- "Actually<RECON>, ..."
- "Another idea<RECON>: ..."
- "Instead<RECON>, I'll try a different category.<LINK>"

# Decision tree (apply top-down per span; first match wins; co-tag only when each genuinely applies, per CC2)

1. Reconsideration cue at sentence start? → <RECON> right after the cue, then continue evaluating the rest of the span.
2. Names a specific candidate? → <INST>, then continue.
3. Justifies why a specific candidate would be tempting? → <PLAUS>.
4. Justifies why a specific candidate is incorrect / contrasts with the answer? → <DISCR>.
5. Describes a student cognitive error with an action verb? → <ERR_DESC>; if step-by-step procedural, → <ERR_SIM>.
6. Selecting / refining / committing to the final distractor set? → <CURATE>.
7. Categorical / design-space statement, no specific candidate named? → <LINK>.
8. Engaging with what the task / question is asking? → <INTER>.
9. Stating / grounding the correct answer? → <CORR>.

If none apply → untagged. Output-formatting talk always falls here (CC3).

# Procedural-meta routing (one-glance summary)

| Function of the sentence                                              | Tag         |
|-----------------------------------------------------------------------|-------------|
| What to generate ("I need three incorrect distractors")               | <INTER>     |
| Design-space constraint on candidates ("must be a neurotransmitter")  | <LINK>      |
| Selection strategy / refinement criterion ("let me pick the distinct ones") | <CURATE> |
| Hedging meta toward committing to the correct answer                  | <CORR>      |
| Output formatting / phrasing of the final list                        | untagged (CC3) |
"""


def extract_examples_from_trace(client: OpenAI, model_config: dict, rt: str, taxonomy: str, max_examples: int = 3) -> dict:
    """Return taxonomy placeholder and a raw examples string for the trace.
    """
    system_prompt = f"""
You are a helper that extracts up to {max_examples} short example spans for each taxonomy label from a single reasoning trace.
For each tag, return a short examples block (plain text) with one line per tag in the following form:
<TAG>: example1; example2
If no examples exist for a tag, use: <TAG>: (none)
Return only the plain text block (no JSON, no commentary).


TAXONOMY:
{taxonomy}
"""
    user_prompt = f"""TRACE START
{rt}
TRACE END

Return only the examples block as described."""
    try:
        examples_raw = prompt_openai(client, system_prompt, user_prompt, model_config)
    except Exception as e:
        print(f"Example extraction failed: {e}")
        return ""

    return examples_raw

def chunk_text(text: str, max_len: int = 2000, min_len: int = 500) -> list:
    """Split text into chunks near max_len using \\n\\n as soft boundaries.
    No overlap. Returns list of tuples: (start_index, chunk_text).
    """
    if len(text) <= max_len:
        return [(0, text)]

    chunks = []
    blocks = text.split("\n\n")

    current = ""
    start = 0
    cursor = 0  # position in original text

    for i, block in enumerate(blocks):
        sep = "\n\n" if i < len(blocks) - 1 else ""
        piece = block + sep

        # Hard split if a single block is too large
        if len(piece) > max_len:
            if current:
                chunks.append((start, current))
                start += len(current)
                current = ""

            offset = 0
            while offset < len(piece):
                chunks.append((cursor + offset, piece[offset:offset + max_len]))
                offset += max_len

            cursor += len(piece)
            start = cursor
            continue

        # If adding this would exceed max_len and we're "big enough", flush
        if current and len(current) + len(piece) > max_len and len(current) >= min_len:
            chunks.append((start, current))
            start += len(current)
            current = piece
        else:
            current += piece

        cursor += len(piece)

    if current:
        chunks.append((start, current))

    return chunks


CHUNK_OPEN = "<<<CHUNK {i}>>>"
CHUNK_CLOSE = "<<<END CHUNK {i}>>>"
_CHUNK_RE = re.compile(r"<<<CHUNK (\d+)>>>(.*?)<<<END CHUNK \1>>>", re.DOTALL)


def _annotate_chunk_batch(client: OpenAI, model_config: dict, chunks_text: list[str], examples_raw: str, taxonomy: str, task_description: str) -> list[str]:
    """Send a batch of chunks in a single prompt and parse out the annotated chunks."""
    wrapped = "\n\n".join(
        f"{CHUNK_OPEN.format(i=i)}\n{c}\n{CHUNK_CLOSE.format(i=i)}" for i, c in enumerate(chunks_text)
    )

    system_prompt = f"""
You are annotating chunks of a thinking-out-loud protocol produced by an expert model with markers of a taxonomy.

Context:
The text is a verbalized reasoning trace of an expert generating incorrect distractor answers for a {task_description} multiple-choice question.

The expert's task was:
"You will be given a {task_description} question. Please generate 3 incorrect distractor answers for the question to be used as multiple-choice options in a multiple-choice exam."

The protocol contains the expert's internal reasoning, planning, and candidate generation steps.
Your job is to annotate each chunk by inserting taxonomy tags. Each marker marks the END of the smallest possible span that instantiates the category.

You will receive several chunks delimited by <<<CHUNK i>>> ... <<<END CHUNK i>>>.
Return ALL chunks back in the SAME order using the SAME delimiters, with annotations inserted into the chunk text. Do not add any commentary outside the delimiters.

TAXONOMY
{taxonomy}

EXAMPLES:
{examples_raw}
"""
    user_prompt = f"""{wrapped}

Return each annotated chunk wrapped in its <<<CHUNK i>>> ... <<<END CHUNK i>>> delimiters, in order, and nothing else.
"""
    try:
        ann = prompt_openai(client, system_prompt, user_prompt, model_config)
    except Exception as e:
        print(f"Batch chunk annotation failed: {e}")
        return list(chunks_text)

    # Parse out chunks by index
    found = {int(idx): body.strip("\n") for idx, body in _CHUNK_RE.findall(ann)}
    out = []
    for i, original in enumerate(chunks_text):
        out.append(found.get(i, original))
    return out


def annotate_rt(client: OpenAI, model_config: dict, rt: str, taxonomy: str = EEDI_TAXONOMY,
                task_description: str = "math",
                fallback_model_config: dict | None = None,
                chunk_max_len: int = 1000, overlap: int = 200,
                chunk_batch_size: int = 4) -> str:
    """Chunk-batched annotator.

    Steps:
      1) extract short examples for each label from the full trace
      2) split the trace into chunks (< chunk_max_len)
      3) send chunks in batches of `chunk_batch_size` per model call
      4) concatenate the annotated chunks
    """
    # 1) extract examples
    examples_raw = extract_examples_from_trace(client, model_config, rt, taxonomy, max_examples=3)

    # 2) split into chunks
    chunks = chunk_text(rt, max_len=chunk_max_len)
    chunk_bodies = [c for _, c in chunks]

    # 3) batch
    per_chunk: list[str] = []
    for i in range(0, len(chunk_bodies), chunk_batch_size):
        batch = chunk_bodies[i:i + chunk_batch_size]
        annotated = _annotate_chunk_batch(client, model_config, batch, examples_raw, taxonomy, task_description)
        per_chunk.extend(annotated)

    return "\n\n".join(per_chunk)

In [ ]:
def annotate_and_save(args):
    client, model_config, fallback_model_config, reasoning, out_path, taxonomy, task_description = args
    try:
        annotator = model_config["model"]
        answer = annotate_rt(client, model_config, reasoning, taxonomy=taxonomy, task_description=task_description)
        if len(answer) < 10:
            print(f"Retrying with chat model because reasoning model most likely ran out of space...")
            annotator = fallback_model_config["model"]
            answer = annotate_rt(client, fallback_model_config, reasoning, taxonomy=taxonomy, task_description=task_description)

        with open(out_path, "w+") as f:
            f.write(f"ANNOTATOR: {annotator}\n")
            f.write(answer)
    except Exception as e:
        print(e)
    return out_path


def annotate_for_manual_review(data_folder: str, run_name: str, n_samples: int = 2, model_config = gpt_4_1,
                               is_reasoning: bool = True, taxonomy: str = EEDI_TAXONOMY,
                               task_description: str = "math"):
    dataset = datasets_by_datafolder[data_folder]

    with open(f"{data_folder}/joint_results/{run_name}_responses_by_datapointid.json", "r") as f:
        responses = json.load(f)

    results_df = pd.read_csv(f"{data_folder}/joint_results/{run_name}_results.csv")

    datapoints = []

    for k,response in responses.items():
        dp = dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")

        trace = response.get("raw_reasoning", "")
        if not is_reasoning:
            trace = response.get("step_by_step")
        datapoints.append((k, dp["Problem"]["Question"], trace, result["proportional_match"], result["number_correct"], result["repetitions"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))

    df = pd.DataFrame(datapoints, columns=["Id", "problem", "trace", "proportional_match", "number_correct", "repetitions", "distractors", "solvable", "nr_steps_of_solution"])

    high_match_solvable_df = df[(df["proportional_match"] > 0.5) & (df["solvable"])]
    low_match_solvable_df = df[(df["proportional_match"] < 0.5)  & (df["solvable"])]

    client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

    out_dir = f"manual_inspection/tagging_evaluation/joint/{data_folder}/{run_name}"
    os.makedirs(out_dir, exist_ok=True)

    tasks = []
    for category, df_cat in {
        "high_match_solvable": high_match_solvable_df,
        "low_match_solvable": low_match_solvable_df
    }.items():
        samples = df_cat.sample(n_samples)
        for i, (_, sample) in enumerate(samples.iterrows()):
            out_path = f"{out_dir}/{category}_{i}.txt"
            tasks.append((client, model_config, model_config, sample["trace"], out_path, taxonomy, task_description)) # always use chat

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(annotate_and_save, tasks), total=len(tasks)))

In [ ]:
def annotate_reasoning(args):
    client, reasoning, model_config, taxonomy, task_description = args

    try:
        annotator = model_config.get("model")
        answer = annotate_rt(client, model_config, reasoning, taxonomy=taxonomy, task_description=task_description)
    except Exception as e:
        print(e)
        return ("", "")

    return (annotator, answer)

# Function to process a DataFrame and generate annotations in parallel
def generate_annotations_and_save(client, model_config, data_folder, run_name, df, category,
                                  taxonomy, task_description, n_samples=60):
    """Generates annotations for `n_samples` traces from `df` and saves them."""
    samples = df.sample(n=min(n_samples, len(df)), random_state=42)
    results = []

    args_list = [(client, trace, model_config, taxonomy, task_description) for trace in samples["trace"]]
    with ThreadPoolExecutor(max_workers=8) as executor:
        res_ann = list(executor.map(annotate_reasoning, args_list))

    for idx, row in enumerate(samples.itertuples(index=False)):
        row_dict = row._asdict() if hasattr(row, '_asdict') else dict(zip(samples.columns, row))
        row_dict["annotation"] = res_ann[idx][1]
        row_dict["annotater"] = res_ann[idx][0]
        results.append(row_dict)
    out_df = pd.DataFrame(results)
    out_dir = f"{data_folder}/joint_results/annotated"
    os.makedirs(out_dir, exist_ok=True)
    out_path = f"{out_dir}/{run_name}_{category}_annot.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved {len(out_df)} annotations to {out_path}")

def annotate_full(data_folder: str, run_name: str, n_samples: int = 60, is_reasoning: bool = True,
                  taxonomy: str = EEDI_TAXONOMY, task_description: str = "math"):
    dataset = datasets_by_datafolder[data_folder]

    with open(f"{data_folder}/joint_results/{run_name}_responses_by_datapointid.json", "r") as f:
        responses = json.load(f)

    results_df = pd.read_csv(f"{data_folder}/joint_results/{run_name}_results.csv")

    datapoints = []

    for k,response in responses.items():
        dp = dataset[int(k)]
        try:
            result = results_df[results_df["Id"] == int(k)].iloc[0]
        except:
            print(f"Could not find {k} in {set(results_df['Id'])}")

        trace = response.get("raw_reasoning", "")
        if not is_reasoning:
            trace = response.get("step_by_step")
        datapoints.append((k, dp["Problem"]["Question"], trace, result["proportional_match"], result["number_correct"], result["repetitions"], result["distractors"], dp["Problem"]["Solvable"], result["num_cor_sol_steps_by_datapointid"]))

    df = pd.DataFrame(datapoints, columns=["Id", "problem", "trace", "proportional_match", "number_correct", "repetitions", "distractors", "solvable", "nr_steps_of_solution"])

    high_match_solvable_df = df[(df["proportional_match"] > 0.5) & (df["solvable"])]
    low_match_solvable_df = df[(df["proportional_match"] < 0.5)  & (df["solvable"])]

    model_config = gpt_4_1
    client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

    generate_annotations_and_save(client, model_config, data_folder, run_name, high_match_solvable_df, "high_match_solvable", taxonomy, task_description, n_samples=n_samples)
    generate_annotations_and_save(client, model_config, data_folder, run_name, low_match_solvable_df, "low_match_solvable", taxonomy, task_description, n_samples=n_samples)

In [ ]:
def load_manual_annotations(dirpath: str):
    """Load all .txt files in dirpath and extract tags with optional +/- prefixes.
    Returns dict: filepath -> list of matches {'pos','sign','label','file'}
    """
    pattern = re.compile(r'<(?P<sign>[+-]?)(?P<label>[A-Z_]+)>')
    files = sorted(glob.glob(str(Path(dirpath) / "*.txt")))
    per_file_matches = {}
    for fp in files:
        try:
            txt = open(fp, 'r', encoding='utf-8').read()
        except Exception as e:
            print(f"Could not read {fp}: {e}")
            continue
        matches = []
        for m in pattern.finditer(txt):
            sign = m.group('sign') or ''
            label = m.group('label')
            matches.append({'pos': m.start(), 'sign': sign, 'label': label, 'file': fp})
        per_file_matches[fp] = matches
    return per_file_matches


def aggregate_counts(per_file_matches, labels):
    counts = {label: {'predicted':0,'fp':0,'fn':0} for label in labels}
    for matches in per_file_matches.values():
        for m in matches:
            label = m['label']
            sign = m['sign']
            if label not in counts:
                counts[label] = {'predicted':0,'fp':0,'fn':0}
            if sign == '-':
                counts[label]['fp'] += 1
            elif sign == '+':
                counts[label]['fn'] += 1
            else:
                counts[label]['predicted'] += 1
    return counts


# Taxonomy labels per dataset (used for agreement metrics and sequence parsing)
EEDI_LABELS = ["INST","INTER","LINK","ERR_DESC","ERR_SIM","RECON","PLAUS","DISCR","CURATE","CORR"]
SCIQ_LABELS = ["INST","INTER","LINK","ERR_DESC","ERR_SIM","RECON","PLAUS","DISCR","CURATE","CORR"]

EEDI_TAG_LABELS = [f"<{l}>" for l in EEDI_LABELS]
SCIQ_TAG_LABELS = [f"<{l}>" for l in SCIQ_LABELS]


def parse_annotation_sequences(glob_pattern: str, tag_labels: list):
    """Parse all annotated CSVs at glob_pattern, append `annotation_sequence`, write `_parsed.csv`."""
    csv_paths = glob.glob(glob_pattern)
    for path in csv_paths:
        df = pd.read_csv(path)
        sequence_lists = []
        for annotation in df["annotation"].fillna(""):
            matches = []
            for tag in tag_labels:
                for m in re.finditer(re.escape(tag), annotation):
                    matches.append((m.start(), tag.strip("<>").lower()))
            matches.sort(key=lambda x: x[0])
            sequence_lists.append(matches)
        df["annotation_sequence"] = [str(seq) for seq in sequence_lists]
        out_path = path.replace("_annot.csv", "_annot_parsed.csv")
        df.to_csv(out_path, index=False)
        print(f"Exported parsed file: {out_path}")


## Eedi


### Annotate a Subset for Manual Review


In [ ]:
annotate_for_manual_review("eedi_data", "deepseek-naive-deepseek-reasoner", n_samples=4)
annotate_for_manual_review("eedi_data", "deepseek-naive-cot-deepseek-chat", n_samples=4, is_reasoning=False)
annotate_for_manual_review("eedi_data", "openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=4)
annotate_for_manual_review("eedi_data", "openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=4, is_reasoning=False)

### Agreement with Manual Annotation


In [ ]:
DIRPATHS_AND_LABELS = [
    ("manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-deepseek-reasoner", EEDI_LABELS),
    ("manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-cot-deepseek-chat", EEDI_LABELS),
    ("manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-z-ai_glm-4.7-reasoner", EEDI_LABELS),
    ("manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-cot-z-ai_glm-4.7-chat", EEDI_LABELS),
]

for DIRPATH, LABELS in DIRPATHS_AND_LABELS:
    print("-"*30)
    print(DIRPATH)
    print("-"*30)

    if not Path(DIRPATH).exists():
        print(f"(skipping, does not exist)")
        continue

    per_file = load_manual_annotations(DIRPATH)
    counts = aggregate_counts(per_file, LABELS)

    rows = []
    total_tp = total_fp = total_fn = 0
    for label in LABELS:
        pred = counts.get(label, {}).get('predicted', 0)
        fp = counts.get(label, {}).get('fp', 0)
        fn = counts.get(label, {}).get('fn', 0)
        tp = max(pred - fp, 0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else None
        recall = tp / (tp + fn) if (tp + fn) > 0 else None
        rows.append({'label':label, 'predicted':pred, 'fp':fp, 'fn':fn, 'tp':tp, 'precision':precision, 'recall':recall})
        total_tp += tp
        total_fp += fp
        total_fn += fn

    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else None
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else None

    df_metrics = pd.DataFrame(rows)
    print(df_metrics.to_string(index=False))
    print()
    print(f"Overall: TP={total_tp} FP={total_fp} FN={total_fn}")
    print(f"Overall precision: {overall_precision:.3f}" if overall_precision is not None else "Overall precision: N/A")
    print(f"Overall recall: {overall_recall:.3f}" if overall_recall is not None else "Overall recall: N/A")

    out_path = Path(DIRPATH) / "aggregate_label_metrics.csv"
    df_metrics.to_csv(out_path, index=False)
    print(f"Saved per-label metrics to {out_path}")


### Full Annotation


In [ ]:
annotate_full("eedi_data", "deepseek-naive-deepseek-reasoner", n_samples=60)
annotate_full("eedi_data", "deepseek-naive-cot-deepseek-chat", n_samples=60, is_reasoning=False)
annotate_full("eedi_data", "openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=60)
annotate_full("eedi_data", "openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=60, is_reasoning=False)

### Parse Annotated Sequences
Parse the annotated sequences into lists of labels.


In [ ]:
parse_annotation_sequences("eedi_data/joint_results/annotated/*_annot.csv", EEDI_TAG_LABELS)


## SciQ


### Annotate a Subset for Manual Review


In [ ]:
annotate_for_manual_review("sciq_data", "deepseek-naive-deepseek-reasoner", n_samples=4,
                           taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_for_manual_review("sciq_data", "deepseek-naive-cot-deepseek-chat", n_samples=4, is_reasoning=False,
                           taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_for_manual_review("sciq_data", "openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=4,
                           taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_for_manual_review("sciq_data", "openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=4, is_reasoning=False,
                           taxonomy=SCIENCE_TAXONOMY, task_description="science")

### Agreement with Manual Annotation


In [ ]:
DIRPATHS_AND_LABELS = [
    ("manual_inspection/tagging_evaluation/joint/sciq_data/deepseek-naive-deepseek-reasoner", SCIQ_LABELS),
    ("manual_inspection/tagging_evaluation/joint/sciq_data/deepseek-naive-cot-deepseek-chat", SCIQ_LABELS),
    ("manual_inspection/tagging_evaluation/joint/sciq_data/openrouter-naive-z-ai_glm-4.7-reasoner", SCIQ_LABELS),
    ("manual_inspection/tagging_evaluation/joint/sciq_data/openrouter-naive-cot-z-ai_glm-4.7-chat", SCIQ_LABELS),
]

for DIRPATH, LABELS in DIRPATHS_AND_LABELS:
    print("-"*30)
    print(DIRPATH)
    print("-"*30)

    if not Path(DIRPATH).exists():
        print(f"(skipping, does not exist)")
        continue

    per_file = load_manual_annotations(DIRPATH)
    counts = aggregate_counts(per_file, LABELS)

    rows = []
    total_tp = total_fp = total_fn = 0
    for label in LABELS:
        pred = counts.get(label, {}).get('predicted', 0)
        fp = counts.get(label, {}).get('fp', 0)
        fn = counts.get(label, {}).get('fn', 0)
        tp = max(pred - fp, 0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else None
        recall = tp / (tp + fn) if (tp + fn) > 0 else None
        rows.append({'label':label, 'predicted':pred, 'fp':fp, 'fn':fn, 'tp':tp, 'precision':precision, 'recall':recall})
        total_tp += tp
        total_fp += fp
        total_fn += fn

    overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else None
    overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else None

    df_metrics = pd.DataFrame(rows)
    print(df_metrics.to_string(index=False))
    print()
    print(f"Overall: TP={total_tp} FP={total_fp} FN={total_fn}")
    print(f"Overall precision: {overall_precision:.3f}" if overall_precision is not None else "Overall precision: N/A")
    print(f"Overall recall: {overall_recall:.3f}" if overall_recall is not None else "Overall recall: N/A")

    out_path = Path(DIRPATH) / "aggregate_label_metrics.csv"
    df_metrics.to_csv(out_path, index=False)
    print(f"Saved per-label metrics to {out_path}")


### Full Annotation


In [ ]:
annotate_full("sciq_data", "deepseek-naive-deepseek-reasoner", n_samples=60,
              taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_full("sciq_data", "deepseek-naive-cot-deepseek-chat", n_samples=60, is_reasoning=False,
              taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_full("sciq_data", "openrouter-naive-z-ai_glm-4.7-reasoner", n_samples=60,
              taxonomy=SCIENCE_TAXONOMY, task_description="science")
annotate_full("sciq_data", "openrouter-naive-cot-z-ai_glm-4.7-chat", n_samples=60, is_reasoning=False,
              taxonomy=SCIENCE_TAXONOMY, task_description="science")

### Parse Annotated Sequences
Parse the annotated sequences into lists of labels.


In [ ]:
parse_annotation_sequences("sciq_data/joint_results/annotated/*_annot.csv", SCIQ_TAG_LABELS)


# Latex

In [ ]:
# LaTeX table: per-(Model, Prompt, Strategy, Dataset) #predicted / Precision / Recall
# Pulls metrics from manual_inspection/tagging_evaluation/joint/{eedi,sciq}_data/<dir>/

STRATEGY_ORDER = [
    ("Task Interpretation",   "INTER"),
    ("Correct Answer Ref.",   "CORR"),
    ("Conceptual Link",       "LINK"),
    ("Error Description",     "ERR_DESC"),
    ("Error Simulation",      "ERR_SIM"),
    ("Outcome Instantiation", "INST"),
    ("Plausibility Check",    "PLAUS"),
    ("Discriminability Check","DISCR"),
    ("Final Set Curation",    "CURATE"),
    ("Reconsideration",       "RECON"),
]

MODEL_PROMPT_DIRS = [
    ("DeepSeek", "CoT",       "deepseek-naive-cot-deepseek-chat"),
    ("DeepSeek", "Reasoning", "deepseek-naive-deepseek-reasoner"),
    ("GLM",      "CoT",       "openrouter-naive-cot-z-ai_glm-4.7-chat"),
    ("GLM",      "Reasoning", "openrouter-naive-z-ai_glm-4.7-reasoner"),
]

BASE = "manual_inspection/tagging_evaluation/joint"


def metrics_for(dirpath, labels):
    if not Path(dirpath).exists():
        return None
    per_file = load_manual_annotations(dirpath)
    counts = aggregate_counts(per_file, labels)
    out = {}
    for label in labels:
        pred = counts.get(label, {}).get('predicted', 0)
        fp = counts.get(label, {}).get('fp', 0)
        fn = counts.get(label, {}).get('fn', 0)
        tp = max(pred - fp, 0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else None
        recall = tp / (tp + fn) if (tp + fn) > 0 else None
        out[label] = (pred, precision, recall)
    return out


def fmt_pct(v):
    return f"{v:.2f}" if v is not None else "--"


def fmt_count(v):
    return f"{v}" if v is not None else "--"


results = {}
for model, prompt, subdir in MODEL_PROMPT_DIRS:
    results[(model, prompt, "eedi")] = metrics_for(f"{BASE}/eedi_data/{subdir}", EEDI_LABELS) or {}
    results[(model, prompt, "sciq")] = metrics_for(f"{BASE}/sciq_data/{subdir}", SCIQ_LABELS) or {}


def emit_strategy_cells(model, prompt, tag):
    eedi = results[(model, prompt, "eedi")].get(tag, (None, None, None))
    sciq = results[(model, prompt, "sciq")].get(tag, (None, None, None))
    return (f"{fmt_count(eedi[0])} & {fmt_pct(eedi[1])} & {fmt_pct(eedi[2])} & "
            f"{fmt_count(sciq[0])} & {fmt_pct(sciq[1])} & {fmt_pct(sciq[2])}")


lines = []
lines.append(r"\begin{table*}[!htbp]")
lines.append(r"\centering")
lines.append(r"\small")
lines.append(r"\begin{tabular}{ll|l||ccc|ccc}")
lines.append(r"\hline")
lines.append(r"\textbf{Model} & \textbf{Prompt} & \textbf{Strategy} &"
             r" \multicolumn{3}{c|}{\textbf{Eedi}} & \multicolumn{3}{c}{\textbf{SciQ}} \\")
lines.append(r"\cline{4-9}")
lines.append(r"& & & \textbf{\#} & \textbf{Precision} & \textbf{Recall}"
             r" & \textbf{\#} & \textbf{Precision} & \textbf{Recall} \\")
lines.append(r"\hline")

for model in ["DeepSeek", "GLM"]:
    lines.append(r"\multirow{20}{*}{\textbf{" + model + r"}}")
    for p_idx, prompt in enumerate(["CoT", "Reasoning"]):
        for s_idx, (display, tag) in enumerate(STRATEGY_ORDER):
            if s_idx == 0:
                prefix = r"& \multirow{10}{*}{\textbf{" + prompt + r"}}"
            else:
                prefix = "&"
            row = prefix + f" & {display} & " + emit_strategy_cells(model, prompt, tag) + r" \\"
            lines.append(row)
        if p_idx == 0:
            lines.append(r"\cline{2-9}")
    lines.append(r"\hline")

lines.append(r"\end{tabular}")
lines.append(r"\caption{Annotation taxonomy distribution across models, prompts, strategies, and datasets.}")
lines.append(r"\label{tab:annotation_taxonomy}")
lines.append(r"\end{table*}")

print("\n".join(lines))
# -> feeds Table 14 (Appendix)


In [ ]:
# Aggregate annotator-vs-manual stats for the paper text.
# Reports per dataset:
#   - Micro overall precision/recall (sum of tp/fp/fn across all 4 dirs)
#   - Macro overall (mean of per-tag micro precision/recall)
#   - Per-tag (precision, recall) micro across all 4 dirs

DATASET_DIRS = {
    "Eedi": [
        ("manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-deepseek-reasoner", EEDI_LABELS),
        ("manual_inspection/tagging_evaluation/joint/eedi_data/deepseek-naive-cot-deepseek-chat", EEDI_LABELS),
        ("manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-z-ai_glm-4.7-reasoner", EEDI_LABELS),
        ("manual_inspection/tagging_evaluation/joint/eedi_data/openrouter-naive-cot-z-ai_glm-4.7-chat", EEDI_LABELS),
    ],
    "SciQ": [
        ("manual_inspection/tagging_evaluation/joint/sciq_data/deepseek-naive-deepseek-reasoner", SCIQ_LABELS),
        ("manual_inspection/tagging_evaluation/joint/sciq_data/deepseek-naive-cot-deepseek-chat", SCIQ_LABELS),
        ("manual_inspection/tagging_evaluation/joint/sciq_data/openrouter-naive-z-ai_glm-4.7-reasoner", SCIQ_LABELS),
        ("manual_inspection/tagging_evaluation/joint/sciq_data/openrouter-naive-cot-z-ai_glm-4.7-chat", SCIQ_LABELS),
    ],
}


def _pr(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else None
    r = tp / (tp + fn) if (tp + fn) > 0 else None
    return p, r


for dataset_name, dir_specs in DATASET_DIRS.items():
    print("=" * 60)
    print(dataset_name)
    print("=" * 60)

    all_labels = []
    for _, labels in dir_specs:
        for l in labels:
            if l not in all_labels:
                all_labels.append(l)

    agg = {l: {"tp": 0, "fp": 0, "fn": 0} for l in all_labels}
    total_tp = total_fp = total_fn = 0

    for dirpath, labels in dir_specs:
        csv_path = Path(dirpath) / "aggregate_label_metrics.csv"
        if not csv_path.exists():
            print(f"(skipping missing: {csv_path})")
            continue
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            l = row["label"]
            if l not in agg:
                continue
            tp = int(row["tp"]); fp = int(row["fp"]); fn = int(row["fn"])
            agg[l]["tp"] += tp
            agg[l]["fp"] += fp
            agg[l]["fn"] += fn
            total_tp += tp
            total_fp += fp
            total_fn += fn

    overall_p, overall_r = _pr(total_tp, total_fp, total_fn)
    op = f"{overall_p:.2f}" if overall_p is not None else "N/A"
    orc = f"{overall_r:.2f}" if overall_r is not None else "N/A"
    print(f"Micro overall: TP={total_tp} FP={total_fp} FN={total_fn}")
    print(f"  precision={op}  recall={orc}")
    print()
    print("Per-tag (precision, recall):")
    footnote_parts = []
    p_list, r_list = [], []
    for l in all_labels:
        p, r = _pr(agg[l]["tp"], agg[l]["fp"], agg[l]["fn"])
        if p is None and r is None:
            print(f"  {l}: no data")
            continue
        p_s = f"{p:.2f}" if p is not None else "N/A"
        r_s = f"{r:.2f}" if r is not None else "N/A"
        tp_v, fp_v, fn_v = agg[l]["tp"], agg[l]["fp"], agg[l]["fn"]
        print(f"  {l}: ({p_s}, {r_s})  [tp={tp_v} fp={fp_v} fn={fn_v}]")
        l_tex = l.replace("_", r"\_")
        footnote_parts.append(f"\\texttt{{{l_tex}}}: ({p_s}, {r_s})")
        if p is not None:
            p_list.append(p)
        if r is not None:
            r_list.append(r)
    macro_p = sum(p_list) / len(p_list) if p_list else None
    macro_r = sum(r_list) / len(r_list) if r_list else None
    mp = f"{macro_p:.2f}" if macro_p is not None else "N/A"
    mr = f"{macro_r:.2f}" if macro_r is not None else "N/A"
    print()
    print(f"Macro (avg of per-tag scores, {len(p_list)} tags with data): precision={mp}  recall={mr}")
    print()
    print("Footnote-ready string:")
    print("; ".join(footnote_parts) + ".")
    print()

# Overall across BOTH datasets (micro = pool tp/fp/fn; macro = avg of per-tag scores)
print("=" * 60)
print("OVERALL (Eedi + SciQ)")
print("=" * 60)
all_labels_all = []
for dir_specs in DATASET_DIRS.values():
    for _, labels in dir_specs:
        for l in labels:
            if l not in all_labels_all:
                all_labels_all.append(l)

agg_all = {l: {"tp": 0, "fp": 0, "fn": 0} for l in all_labels_all}
tot_tp = tot_fp = tot_fn = 0
for dir_specs in DATASET_DIRS.values():
    for dirpath, labels in dir_specs:
        csv_path = Path(dirpath) / "aggregate_label_metrics.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            l = row["label"]
            if l not in agg_all:
                continue
            tp = int(row["tp"]); fp = int(row["fp"]); fn = int(row["fn"])
            agg_all[l]["tp"] += tp
            agg_all[l]["fp"] += fp
            agg_all[l]["fn"] += fn
            tot_tp += tp; tot_fp += fp; tot_fn += fn

mp_o, mr_o = _pr(tot_tp, tot_fp, tot_fn)
print(f"Micro overall: TP={tot_tp} FP={tot_fp} FN={tot_fn}  precision={mp_o:.2f}  recall={mr_o:.2f}")

p_list, r_list = [], []
for l in all_labels_all:
    p, r = _pr(agg_all[l]["tp"], agg_all[l]["fp"], agg_all[l]["fn"])
    if p is not None:
        p_list.append(p)
    if r is not None:
        r_list.append(r)
macro_p = sum(p_list) / len(p_list) if p_list else None
macro_r = sum(r_list) / len(r_list) if r_list else None
mp = f"{macro_p:.2f}" if macro_p is not None else "N/A"
mr = f"{macro_r:.2f}" if macro_r is not None else "N/A"
print(f"Macro (avg of per-tag scores, {len(p_list)} tags with data): precision={mp}  recall={mr}")
# -> headline 0.97/0.95 macro precision/recall quoted in Section 4.2 ("Annotation" paragraph)
